# MEMORIA TÉCNICA
# Nutriscore 2.0: Sistema Inteligente de Clasificación Alimentaria

**Autor:** Mikel Añibarro Ortega  
**Fecha:** Mayo 2026  
**Bootcamp:** Data Science - The Bridge, Campus Bilbao

---

# I. INTRODUCCIÓN

## 🎯 Contexto del Problema

En 2026, **Millones de alimentos** están disponibles en mercados europeos, pero los consumidores solo tienen **UN etiquetado global** para clasificarlos: **Nutriscore**.

### El Problema

Nutriscore solo analiza **nutrientes**, ignorando dos aspectos críticos:

1. **Grado de procesamiento** (NOVA)
2. **Seguridad de aditivos químicos** (651 aditivos permitidos en la UE)

**Resultado:** Un usuario ve "Nutriscore A" (saludable) pero el alimento puede contener aditivos peligrosos.

---

## 🎓 Objetivos del Proyecto

1. **Clasificar 651 aditivos** según riesgo científico basado en **PubMed** (Scientific Safety Index - SSI)
2. **Segmentar 836k alimentos** en grupos homogéneos usando **K-Means**
3. **Validar clustering** con **Hierarchical Clustering** (WARD)
4. **Crear clasificación integrada** (Nutriscore + NOVA + SSI) en **4 clusters**
5. **Desplegar en app Streamlit** accesible para consumidores

### Resultado Final

**Nutriscore 2.0:** Una clasificación holística que combina:
- Calidad nutricional
- Grado de procesamiento  
- Seguridad de aditivos basada en ciencia

---

# II. DATASET

## 📊 Fuentes de Datos

### 1. Dataset de Aditivos

| Métrica | Valor |
|---------|-------|
| Origen | Open Food Facts + EFSA oficial |
| Aditivos totales | 651 |
| Variables | Código E, nombre químico, status EFSA |
| Completitud | 100% |

### 2. Dataset de Alimentos

| Métrica | Valor |
|---------|-------|
| Origen | Open Food Facts |
| Alimentos | 836,897 |
| Features | Nutriscore, NOVA, aditivos (one-hot) |
| Completitud | 100% (después de limpieza) |

---

## 📈 Análisis Exploratorio (EDA)

### Distribución Nutriscore

```
A (1): 126,005  (15.0%) ✅ Mejor
B (2):  92,342  (11.0%)
C (3): 177,776  (21.2%)
D (4): 202,284  (24.2%)
E (5): 239,141  (28.6%) ❌ Peor
```

**Conclusión:** 53% de alimentos tienen Nutriscore D-E (pobre nutrición)

### Distribución NOVA

```
NOVA 1 (Natural):        15-20%
NOVA 2 (Procesados):     25-30%
NOVA 3-4 (Ultraprocesados): 55-60% ⚠️
```

**Conclusión:** Mayoría de alimentos son ultraprocesados

### Aditivos por Alimento

```
0 aditivos:     20-25%
1-3 aditivos:   40-45%
4-8 aditivos:   25-30%
8+ aditivos:    5-10% ⚠️ (ultraprocesados)
```

**Conclusión:** Fuerte correlación entre NOVA y cantidad de aditivos

---

# III. PREPROCESAMIENTO DE LOS DATOS

## 🔧 Verificación de Calidad

### Aditivos

```
✅ Sin duplicados por código E
✅ 100% con status EFSA validado
✅ Nombres químicos estandarizados
✅ Preparado para búsquedas PubMed
```

### Alimentos

```
✅ Nutriscore: 100% entre 1-5
✅ NOVA: 100% entre 1-4
✅ Aditivos: Códigos E estandarizados
✅ Sin valores faltantes en features críticas
```

---

## 🔄 Transformaciones Aplicadas

### 1. Estandarización Nutriscore

```python
A → 1  (excelente)
B → 2  (bueno)
C → 3  (normal)
D → 4  (malo)
E → 5  (muy malo)
```

### 2. One-Hot Encoding de Aditivos

**Antes:**
```
product_name | additives
Yogur        | E100, E101, E102
```

**Después:**
```
product_name | E100 | E101 | E102 | ...
Yogur        |  1   |  1   |  1   | ...
```

### 3. Cálculo de Total Aditivos

```python
total_aditivos = suma de all E### presentes
```

### 4. StandardScaler para K-Means

```python
# Normalizar features a media=0, desv=1
X_scaled = StandardScaler().fit_transform(X)

# ¿Por qué? Evitar que total_aditivos (0-20+)
# domine sobre Nutriscore (1-5)
```

---

# IV. MODELADO

## 🤖 Etapa 1: Clasificación de Aditivos (SSI)

### Metodología

1. **Búsqueda en PubMed:** 651 aditivos × 16 palabras clave = 10,400 búsquedas
2. **Conteo de papers:** Por categoría (negativos, positivos, por tipo de estudio)
3. **Filtros inteligentes:**
   - Negaciones: Si hay "no dañino" reduce peso de "adverso"
   - Ponderación estudio: In vitro ×0.30, animal ×0.50, humano ×1.00
4. **Cálculo SSI:** Fórmula que combina evidencia + recencia + confianza
5. **Validación EFSA:** Si EFSA retira aditivo, clasificar como EVITABLE

### Resultado: 3 Categorías de Aditivos

```
🟢 SEGURO:      475 aditivos (73%)
🟡 PRECAUCIÓN:   84 aditivos (13%)
🔴 EVITABLE:     92 aditivos (14%)
```

---

## 🎯 Etapa 2: Clustering de Alimentos (K-Means)

### Features Seleccionados

```python
features = [
    'nutriscore_grade',    # Calidad nutricional (1-5)
    'nova_group',          # Procesamiento (1-4)
    'total_aditivos'       # Cantidad de aditivos (0-20+)
]
```

**¿Por qué estos 3?**
- Representan las 3 dimensiones clave de riesgo
- No colineales
- Escalables

### Determinación de K Óptimo

**Método 1: Elbow Method**

![Elbow Plot](img/metodo_codo_alimentos2.png)

✅ Codo claro en K=4

**Método 2: Silhouette Score**

![Silhouette Plot](img/silhouette_4_clusters2.png)

```
K=4 → Silhouette Score = 0.41 (Bueno)
```

### Conclusión: K=4 es ÓPTIMO

---

## ✅ Etapa 3: Validación con Hierarchical Clustering

### Dendrograma

![Dendrograma](img/dendrograma_ward_validacion.png)

**Observación:** En altura ~53, se observan EXACTAMENTE 4 ramas principales

### Convergencia de Métodos

```
K-Means (determinístico)      → 4 clusters
            +
Hierarchical (exploratorio)   → 4 ramas naturales
            =
✅ CONFIANZA MÁXIMA EN K=4
```

**Conclusión:** K=4 es robusto, natural y reproducible

---

# V. PREDICCIÓN Y RESULTADOS FINALES

## 📊 Los 4 Clusters de Alimentos

### Matriz de Perfiles

![Matriz Perfiles](img/matriz2.png)

### Cluster 0: "Falso Saludable" (21.1%)

```
Nutriscore:  2.28 (parece bueno)
NOVA:        3.65 (pero ultraprocesado)
Aditivos:    1.39 (pocos)
Tamaño:      176,894 alimentos

Ejemplos: Zumos "naturales", yogures de dieta
Riesgo: ENGAÑOSO - parece saludable pero no lo es
```

### Cluster 1: "Simple Malo" (41.2%)

```
Nutriscore:  4.54 (malo)
NOVA:        3.67 (procesado)
Aditivos:    1.74 (pocos)
Tamaño:      344,688 alimentos

Ejemplos: Refrescos, snacks salados
Riesgo: CLARO pero al menos son "simples"
```

### Cluster 2: "Verdaderamente Saludable" (15.6%)

```
Nutriscore:  1.93 (excelente)
NOVA:        1.17 (natural)
Aditivos:    0.06 (CASI NINGUNO) ✅✅
Tamaño:      130,602 alimentos

Ejemplos: Frutas, verduras frescas
Riesgo: MÍNIMO
```

### Cluster 3: "Ultraprocesado" (22.1%)

```
Nutriscore:  4.18 (malo)
NOVA:        4.00 (ultraprocesado)
Aditivos:    8.97 (MUCHOS) ⚠️⚠️
Tamaño:      184,543 alimentos

Ejemplos: Productos ultraprocesados industriales
Riesgo: MÁXIMO
```

---

## 📈 Visualización 3D

![Scatter 3D](img/dispersion.png)

**Interpretación:**
- Cada punto = 1 alimento
- X = Nutriscore, Y = NOVA, Z = Aditivos
- Colores = Clusters asignados por K-Means
- Separación clara entre clusters

---

## 🎓 Solución Final: Nutriscore 2.0

### Clasificación Integrada

```
Para cada alimento:

Nutriscore (1-5)      ← De Open Food Facts
    +
NOVA (1-4)            ← De Open Food Facts
    +
Total aditivos        ← Contado del CSV
    +
Aditivos por tipo     ← Mapeado con SSI
    =
NUTRISCORE 2.0
└─ Cluster (0-3)
└─ Aditivo dominante (SEGURO/PRECAUCIÓN/EVITABLE)

```
![nutriscore 2.0](img/nutriscore2-0.png)

### 12 Categorías Finales

```
0_SEGUROS        0_PRECAUCION      0_EVITAR
1_SEGUROS        1_PRECAUCION      1_EVITAR
2_SEGUROS        2_PRECAUCION      2_EVITAR
3_SEGUROS        3_PRECAUCION      3_EVITAR
```

**Cada categoría agrupa alimentos con:
- Características similares (cluster)
- Riesgo de aditivos similar

---

## 💡 Impacto en el Negocio

### Para Consumidores

✅ Información COMPLETA en 1 clasificación  
✅ Decisiones informadas basadas en ciencia  
✅ Acceso fácil via app Streamlit  
✅ Comparación entre productos  

### Para Industria

✅ Incentivo para reducir aditivos  
✅ Presión competitiva positiva  
✅ Diferencial marketing ("Cluster 2 certified")  

### Para Reguladores

✅ Herramienta de monitoreo científico  
✅ Datos para revisar aditivos  
✅ Apoyo a políticas de salud pública  

---

# VI. CONCLUSIONES Y FUTUROS PASOS

## ✅ Fortalezas del Proyecto

### 1. Metodología Robusta

```
✅ SSI basado en evidencia científica real (PubMed)
✅ K-Means validado con Hierarchical Clustering
✅ Dos métodos independientes convergen en K=4
✅ Reproducible y escalable
```

### 2. Integración Holística

```
✅ Combina 3 dimensiones (nutrientes + procesamiento + aditivos)
✅ Supera limitaciones de Nutriscore clásico
✅ Ofrece visión 360° del alimento
```

### 3. Accesibilidad

```
✅ App Streamlit user-friendly
✅ Logo profesional
✅ Información clara y comprensible
```

---

## ⚠️ Limitaciones y Debilidades

### 1. Completitud de Datos

```
❌ No todos los alimentos en OFF tienen aditivos listados
✅ Solución: Usar datos de 836k con información completa
```

### 2. Sinergia de Aditivos

```
❌ SSI analiza aditivos individualmente
❌ No considera interacciones entre aditivos
✅ Futura mejora: Estudiar combinaciones peligrosas
```

### 3. Cambio Regulatorio

```
❌ Si EFSA retira/aprueba aditivo, SSI cambia
✅ Solución: Crear pipeline a tiempo real
```

---

## 🚀 Propuestas de Futuras Mejoras

### Expansión 

```
✅ Análisis de interacciones de aditivos
✅ ML predictivo: predecir cambios de cluster si se modifica formulación
✅ Incorporar datos de ingredientes base (azúcares, grasas saturadas)
✅ Integración con apps de supermercado
✅ Propuesta como estándar europeo (EFSA)
```

---


## 🎯 Conclusión Final

**Nutriscore 2.0 no es la solución final, es el COMIENZO.**

Hemos demostrado que:

✅ Es **técnicamente posible** combinar nutrientes + procesamiento + aditivos  
✅ La **estructura natural** de datos apoya 4 clusters distintos  
✅ El sistema es **robusto, reproducible y escalable**  
✅ Existe **demanda real** de esta información  

El siguiente paso es llevar esto a producción y validar el impacto real en salud de consumidores.

---

Autor: Mikel Añibarro Ortega